# Entrainement avec seulement features importances

In [1]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import utils
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [3]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

numeric_features = data.select_dtypes(include=['int64', 'float64'])
categorical_features = ['statut_marital', 'departement', 'poste', 'domaine_etude']

preprocessor = ColumnTransformer(
    transformers=[
        ('standard_scaler', StandardScaler(), make_column_selector(dtype_include='number')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_include='object')),
    ],
    remainder='drop'
)

In [15]:
from sklearn.feature_selection import SelectFromModel
import numpy as np

feature_selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    max_features=10,
    threshold=-np.inf
)

final_model = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=15,
    random_state=42
)

In [18]:
from sklearn.linear_model import LogisticRegression
from technova_correlation_cleaning import CorrelationFilter
from technova_features import TechNovaFeatureEngineering
from sklearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42, class_weight='balanced')
logistic_regression_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

models = [
    {'name' : 'LogisticRegression', 'model': logistic_regression_model},
    {'name' : 'RandomForestClassifier', 'model': random_forest_model},
]

for model in models:
    print(f'Training model: {model["name"]}')
    pipeline = Pipeline([
        ('features', TechNovaFeatureEngineering()),
        ('corr_cleaning', CorrelationFilter(threshold=0.80)),
        ('preprocessor', preprocessor),
        ('feature_selection', feature_selector),
        ('model', model['model']),
    ])
    utils.benchmark(pipeline, train_data)

    print(f'------------------------------\n')

Training model: LogisticRegression
--- Validation Fold Results ---
Validation Recall : [0.47368421 0.73684211 0.73684211 0.63157895 0.55263158], Recall moyen : 0.6263157894736842
Validation F1-Scores [0.27906977 0.42424242 0.42105263 0.38095238 0.41176471], F1 moyen : 0.38341638201959316
Validation ROC AUC : [0.61204147 0.7785199  0.76769971 0.70344643 0.71667112], ROC moyen : 0.7156757260016678

--- Train Fold Results (Overfit Check) ---
Train Recall : [0.64473684 0.69078947 0.70394737 0.69736842 0.60526316], Recall moyen : 0.6684210526315789
Train F1-Scores [0.37984496 0.42857143 0.40530303 0.42315369 0.37627812], F1 moyen : 0.40263024626778926
Train ROC AUC : [0.71874165 0.74869922 0.73868488 0.74690648 0.70426423], ROC moyen : 0.7314592901256829
------------------------------

Training model: RandomForestClassifier
--- Validation Fold Results ---
Validation Recall : [0.28947368 0.39473684 0.28947368 0.39473684 0.28947368], Recall moyen : 0.3315789473684211
Validation F1-Scores [0.2